In [1]:
#!pip install pymupdf langchain langchain-community chromadb langchain_huggingface sentence-transformers ollama

In [2]:
# Imports
import pdfplumber
import pickle
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
# from langchain_huggingface import HuggingFaceEmbeddings # Alternative to SentenceTransformerEmbeddings
import ollama
import os

/home/michael/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
rebuild_database = True
folder_path = "./pdfs"   # <-- change to your folder path

In [4]:
def load_pdfs_from_folder(folder_path: str) -> list[tuple[str, str]]:
    """Returns a list of (text, source_filename) tuples for all PDFs in folder."""
    results = []
    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]
    
    if not pdf_files:
        raise ValueError(f"No PDF files found in: {folder_path}")
    
    for filename in pdf_files:
        path = os.path.join(folder_path, filename)
        with pdfplumber.open(path) as pdf:
            text = "\n".join(
                page.extract_text() for page in pdf.pages if page.extract_text()
            )
        results.append((text, filename))
        print(f"  Loaded '{filename}': {len(text):,} characters")
    
    return results

# --- Load all PDFs ---
if rebuild_database:
    PDF_FOLDER = folder_path   # <-- change to your folder path
    print(f"Scanning for PDFs in: {PDF_FOLDER}")
    pdf_docs = load_pdfs_from_folder(PDF_FOLDER)
    print(f"Loaded {len(pdf_docs)} PDF(s)\n")


Scanning for PDFs in: ./pdfs
  Loaded 'Li et al. - 2023 - Bayesian causal inference a critical review.pdf': 91,554 characters
  Loaded 'Liu et al. - 2024 - A Review of Causal Inference Methods for Estimating the Effects of Exposure Change when Incident Exp.pdf': 68,044 characters
  Loaded 'The book of why.pdf': 808,428 characters
  Loaded 'Colnet et al. - 2024 - Causal Inference Methods for Combining Randomized Trials and Observational Studies A Review.pdf': 127,820 characters
  Loaded 'Elements_Of_Causal_Inference.pdf': 505,579 characters
  Loaded 'Niu et al. - 2024 - Comprehensive Review and Empirical Evaluation of Causal Discovery Algorithms for Numerical Data.pdf': 173,047 characters
  Loaded 'Zanga et al. - 2022 - A Survey on Causal Discovery Theory and Practice.pdf': 126,734 characters
  Loaded 'CAUSAL_INFERENCE_AND_DISCOVERY_IN_PYTHON.pdf': 826,690 characters
  Loaded 'Dahabreh and Bibbins-Domingo - 2024 - Causal Inference About the Effects of Interventions From Observational St

In [5]:

if rebuild_database:
    splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=250)
    all_chunks = []
    all_metadatas = []

    for text, filename in pdf_docs:
        chunks = splitter.split_text(text)
        all_chunks.extend(chunks)
        all_metadatas.extend([{"source": filename}] * len(chunks))
        print(f"  '{filename}': {len(chunks)} chunks")

    print(f"\nTotal chunks: {len(all_chunks)}")

    # --- Build vectorstore ---
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
    # embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")  # alternative

    vectorstore = Chroma.from_texts(
        texts=all_chunks,
        embedding=embeddings,
        metadatas=all_metadatas,
        persist_directory=f"{folder_path}/chroma_db"  # Save vectostore to disk for future use
    )   # each chunk tagged with its source PDF
    retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
    print("Vector store ready ✓")
else:
    # Load existing vectorstore from Chroma persistence
    try:
        vectorstore = Chroma(
            persist_directory=f"{folder_path}/chroma_db",
            embedding_function=embeddings
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
        print(f"Vector store loaded from '{folder_path}/chroma_db'")
    except Exception as e:
        print(f"No existing vector store found. Error: {e}")
        print("Please set rebuild_database=True to create it.")

  'Li et al. - 2023 - Bayesian causal inference a critical review.pdf': 41 chunks
  'Liu et al. - 2024 - A Review of Causal Inference Methods for Estimating the Effects of Exposure Change when Incident Exp.pdf': 31 chunks
  'The book of why.pdf': 361 chunks
  'Colnet et al. - 2024 - Causal Inference Methods for Combining Randomized Trials and Observational Studies A Review.pdf': 57 chunks
  'Elements_Of_Causal_Inference.pdf': 226 chunks
  'Niu et al. - 2024 - Comprehensive Review and Empirical Evaluation of Causal Discovery Algorithms for Numerical Data.pdf': 77 chunks
  'Zanga et al. - 2022 - A Survey on Causal Discovery Theory and Practice.pdf': 57 chunks
  'CAUSAL_INFERENCE_AND_DISCOVERY_IN_PYTHON.pdf': 368 chunks
  'Dahabreh and Bibbins-Domingo - 2024 - Causal Inference About the Effects of Interventions From Observational Studies in Medical Journals.pdf': 29 chunks
  'Pearl_2009_Causality.pdf': 613 chunks
  'causal_roadmap.pdf': 33 chunks
  'Squires and Uhler - 2023 - Causal Struc

/tmp/ipykernel_27957/1312854492.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21153.27it/s]


Vector store ready ✓


In [6]:
# Cell 4 — Chat function
def chat(question: str, history: list[dict]) -> str:
    # Retrieve relevant chunks from the PDF
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(d.page_content for d in docs)

    # Build the prompt
    system_prompt = (
        "You are a helpful assistant. Answer the user's question using ONLY "
        "the context below. If the answer is not in the context, say so.\n\n"
        f"CONTEXT:\n{context}"
    )

    # Append this turn to history
    history.append({"role": "user", "content": question})

    response = ollama.chat(
        model="qwen3.5:9b",   # or "gemma:2b", "gemma:7b", etc.
        messages=[{"role": "system", "content": system_prompt}] + history,
        options={"num_ctx": 32768} 
    )

    answer = response["message"]["content"]
    history.append({"role": "assistant", "content": answer})
    return answer

In [7]:
# Cell 5 — Interactive chat loop (run this cell to start chatting)
history = []
print("PDF Chatbot ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue
    answer = chat(question, history)
    print(f"\nLLM: {answer}\n")

PDF Chatbot ready. Type 'quit' to exit.


LLM: Based on the provided context, here is what is known about causality:

**Nature and Definition**
*   Causality has evolved from a "nebulous concept" into a "mathematical theory" with significant applications in statistics, artificial intelligence, economics, philosophy, cognitive science, and the health and social sciences.
*   Judea Pearl presents a comprehensive theory that unifies probabilistic, manipulative, counterfactual, and structural approaches to causation.
*   It involves a "calculus of causation" consisting of two languages: causal diagrams (to express what we know) and a symbolic language resembling algebra (to express what we want to know).
*   Causal relations are unidirectional, going from cause to effect, which distinguishes them from the symmetrical laws of physics.

**The Ladder of Causation**
Causality is described using a metaphor known as the Ladder of Causation, which has three rungs representing different levels of 

In [8]:
model="qwen3.5:9b"
! ollama stop {model}

]11;?\⠙ 